# Belgian Rail Punctuality vs Weather — Exploratory SQL Analysis

**Goal:** explore whether train delays on the Belgian rail network relate to weather conditions (temperature, precipitation), using SQL as the primary analysis tool.

**Data sources:**
- Train punctuality: [Infrabel Open Data — Raw punctuality data (D-1)](https://opendata.infrabel.be/explore/dataset/ruwe-gegevens-van-stiptheid-d-1/)
- Weather: [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api)

**Approach:**
1. Load raw CSVs and push them into a SQLite database (`treni_meteo.db`)
2. Profile the data in SQL: types, ranges, missing values
3. Investigate and resolve data quality questions (delay units, cancellations vs terminus stations)
4. Build a per-train delay summary as a first, minimal analytical output
5. Export the result for further analysis in Power BI

*Note: this is an early iteration working with a single day of data, used to validate the pipeline end-to-end. The same logic will be extended once multiple days of data are available.*

## 1. Setup

In [ ]:
import pandas as pd
import sqlite3

## 2. Load raw data

Train data uses `;` as separator (Infrabel export default). Weather data (Open-Meteo) has a few metadata rows before the actual table, hence `skiprows=3`.

In [ ]:
df_trains = pd.read_csv('data/trains_raw.csv', sep=';')
df_weather = pd.read_csv('data/weather_raw.csv', sep=',', skiprows=3)

print(f"Trains: {df_trains.shape}")
print(f"Weather: {df_weather.shape}")

## 3. Load into SQLite

Both tables are pushed into the same SQLite database so they can be joined with SQL later. `pandas` is used only for ingestion (reading the CSVs); all cleaning and analysis from this point on is done in SQL.

In [ ]:
conn = sqlite3.connect("treni_meteo.db")

df_trains.to_sql("trains", conn, if_exists="replace", index=False)
df_weather.to_sql("weather", conn, if_exists="replace", index=False)

# Sanity check: confirm both tables exist
query = """
SELECT name FROM sqlite_master WHERE type = 'table';
"""
pd.read_sql(query, conn)

## 4. Schema exploration

Quick look at column types for both tables before doing any analysis.

In [ ]:
query = """PRAGMA table_info(trains);"""
pd.read_sql(query, conn)

In [ ]:
query = """PRAGMA table_info(weather);"""
pd.read_sql(query, conn)

## 5. Profiling the delay column

Checking range and missing values in `Delay at arrival` before trusting it for analysis.

In [ ]:
query = """
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN "Delay at arrival" IS NULL THEN 1 ELSE 0 END) AS null_count,
    MIN("Delay at arrival") AS min_delay,
    MAX("Delay at arrival") AS max_delay,
    AVG("Delay at arrival") AS avg_delay
FROM trains;
"""
pd.read_sql(query, conn)

**Finding:** average delay ≈ 98.7. If this were minutes, that would mean every stop averages over 1.5 hours of delay — implausible for a functioning rail network. Interpreted as **seconds** (≈1.6 min average, ≈80 min max), which is realistic. Negative values represent trains arriving *early*.

## 6. Investigating missing delay values

~4.6% of rows have `NULL` in `Delay at arrival`. Before excluding or reinterpreting them, checking whether they represent cancelled trains or something else structural in the dataset.

In [ ]:
query = """
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN "Actual arrival time" IS NULL THEN 1 ELSE 0 END) AS null_actual_arrival,
    SUM(CASE WHEN "Planned arrival time" IS NULL THEN 1 ELSE 0 END) AS null_planned_arrival
FROM trains
WHERE "Delay at arrival" IS NULL;
"""
pd.read_sql(query, conn)

**Finding:** for all 3,407 NULL rows, both `Actual arrival time` **and** `Planned arrival time` are NULL. This is not a cancellation — it means these rows represent a train's **origin station** (departure only, no arrival is ever scheduled there). Confirmed by inspecting individual train journeys: each train has one row per stop, and only the first row (origin) lacks an arrival time.

**Implication:** these rows should be excluded when calculating arrival delays (there's nothing to measure), but they are not evidence of weather-related cancellations. That question remains open and would need a different signal in the data (e.g. missing stops mid-journey).

In [ ]:
# Spot-check: inspect all rows for a single train to see the pattern directly
query = """
SELECT "Train number", "Stopping place", "Planned departure time", "Planned arrival time",
       "Delay at departure", "Delay at arrival"
FROM trains
WHERE "Train number" = (SELECT "Train number" FROM trains WHERE "Delay at arrival" IS NULL LIMIT 1)
ORDER BY "Planned departure time";
"""
pd.read_sql(query, conn)

## 7. Per-train delay summary

Each train appears multiple times in the dataset (once per stop). To avoid pseudo-replication when aggregating, this builds **one row per train**, using its final delay value and a binary `is_delayed` flag based on the EU punctuality standard (>5 minutes / 300 seconds).

*Known simplification: this takes `MAX(Delay at arrival)` per train rather than isolating the true final stop via a window function (`ROW_NUMBER()`), which would be more precise. Flagged here as a next iteration.*

In [ ]:
query = """
SELECT 
    "Train number",
    MAX("Planned arrival time") AS planned_arrival_time,
    CASE
        WHEN MAX("Delay at arrival") > 300 THEN 'YES'
        ELSE 'NO'
    END AS is_delayed
FROM trains
GROUP BY "Train number";
"""
df_result = pd.read_sql(query, conn)
df_result.head()

## 8. Export for Power BI

Exporting the per-train summary as a CSV, to be joined with weather data and visualized in Power BI.

In [ ]:
df_result.to_csv("output/trains_delay_summary.csv", index=False)
print(f"Exported {len(df_result)} rows.")

## Next steps

- Resolve the per-train "final stop" logic properly with `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`
- Join the per-train summary with hourly weather data
- Extend to multiple days once more Infrabel data is collected, to support a real correlation analysis (`CORR()` in SQL)
- Build the Power BI dashboard: % delayed trains by hour vs. temperature